<a href="https://colab.research.google.com/github/gloriaconcepto/Clothing-review-embedding-project/blob/main/Copy_of_Clothing_reviews_embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AI-Powered Customer Review Analytics Platform
Summary

A production-ready customer review analytics platform that applies state-of-the-art embedding models, semantic search, vector databases, and dimensionality reduction to uncover customer insights. The platform supports review categorization, clustering, and intelligent retrieval, providing a strong foundation for recommendation engines, customer support automation, and product analytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Installed libraries dependecies
!pip install -q kaggle

In [ ]:
import os
import shutil

# Create the Kaggle configuration directory
os.makedirs('/root/.kaggle', exist_ok=True)

# Copy the API key
shutil.copy(
    '/content/drive/MyDrive/kaggle/kaggle.json',
    '/root/.kaggle/kaggle.json'
)

# Set the required permissions
os.chmod('/root/.kaggle/kaggle.json', 600)

print("Kaggle API configured successfully!")

Kaggle API configured successfully!


In [ ]:
#  download the data from kaggle..
!kaggle datasets download -d muhammadjiyadkhan/womens-clothing-ecommerce-reviews

Dataset URL: https://www.kaggle.com/datasets/muhammadjiyadkhan/womens-clothing-ecommerce-reviews
License(s): unknown
100% 2.79M/2.79M [00:00<00:00, 200MB/s]



In [ ]:
# unzip
!unzip -q womens-clothing-ecommerce-reviews.zip -d womens_reviews

In [ ]:
# unzip and save to google drive
import os
import shutil

destination = "/content/drive/MyDrive/newKaggleDataSet/WomensClothingReviews"
source = "womens_reviews"
expected_file = "Womens Clothing E-Commerce Reviews.csv"

os.makedirs(destination, exist_ok=True)

if os.path.exists(os.path.join(destination, expected_file)):
    print("✅ Dataset already exists. Skipping copy.")
else:
    shutil.copytree(source, destination, dirs_exist_ok=True)
    print("✅ Dataset copied successfully.")


✅ Dataset already exists. Skipping copy.


In [ ]:
# load the csv
import pandas as pd

original_review_df = pd.read_csv(
    "/content/drive/MyDrive/newKaggleDataSet/WomensClothingReviews/womens_clothing_reviews.csv"
)
print(original_review_df.shape)
original_review_df.head()


(23486, 11)


,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


# FEATURE ENGINEERING

In [ ]:
original_review_df.isna().sum()

,0
Unnamed: 0,0
Clothing ID,0
Age,0
Title,3810
Review Text,845
Rating,0
Recommended IND,0
Positive Feedback Count,0
Division Name,14
Department Name,14


In [ ]:
# Clean the dataset
import pandas as pd

# Make a copy
clean_df = original_review_df.copy()

# Remove duplicate rows
clean_df = clean_df.drop_duplicates()

# Remove rows with missing Review Text
clean_df = clean_df.dropna(subset=["Review Text"])

# Remove rows with missing Rating
clean_df = clean_df.dropna(subset=["Rating"])

# Remove rows with empty Review Text
clean_df = clean_df[
    clean_df["Review Text"].astype(str).str.strip() != ""
]

# Remove very short reviews (<5 words)
clean_df = clean_df[
    clean_df["Review Text"].str.split().str.len() >= 5
]

# Remove duplicate review texts
clean_df = clean_df.drop_duplicates(
    subset=["Review Text"]
)

# Reset index
clean_df = clean_df.reset_index(drop=True)

print(clean_df.shape)

(22600, 11)


In [ ]:
print(clean_df.isna().sum())

print("\nDuplicate reviews:",
      clean_df.duplicated(subset=["Review Text"]).sum())

Unnamed: 0                    0
Clothing ID                   0
Age                           0
Title                      2952
Review Text                   0
Rating                        0
Recommended IND               0
Positive Feedback Count       0
Division Name                13
Department Name              13
Class Name                   13
dtype: int64

Duplicate reviews: 0


In [ ]:
# Save the cleaned dataset
output_path = "/content/drive/MyDrive/newKaggleDataSet/WomensClothingReviews/cleaned_reviews.csv"

clean_df.to_csv(output_path, index=False)

print(f"Saved cleaned dataset to:\n{output_path}")

Saved cleaned dataset to:
/content/drive/MyDrive/newKaggleDataSet/WomensClothingReviews/cleaned_reviews.csv


In [ ]:
!pip -q install --upgrade openai chromadb tiktoken
!pip -q install scikit-learn matplotlib pandas numpy python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6

In [ ]:
import openai
import chromadb
import sklearn
import pandas

print("Environment Ready ✅")

Environment Ready ✅


# Project Configuration

In [ ]:
from pathlib import Path

# Root project folder
PROJECT_ROOT = Path("/content/drive/MyDrive/AIEngineering/WomenClothReview")

# Dataset folder
DATA_DIR = PROJECT_ROOT / "datasets"

# Output folder
OUTPUT_DIR = PROJECT_ROOT / "outputs"

# Chroma database
VECTOR_DB_DIR = PROJECT_ROOT / "vectordb"

# Figures
FIGURES_DIR = OUTPUT_DIR / "figures"

# Create folders automatically
for folder in [
    DATA_DIR,
    OUTPUT_DIR,
    VECTOR_DB_DIR,
    FIGURES_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)

/content/drive/MyDrive/AIEngineering/WomenClothReview


### Configure OpenAI

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.manifold import TSNE
from scipy.spatial import distance

import chromadb

from openai import OpenAI
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

warnings.filterwarnings("ignore")

In [ ]:
from getpass import getpass

OPENAI_API_KEY = getpass("Enter OpenAI API Key: ")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

client = OpenAI()

EMBEDDING_MODEL = "text-embedding-3-small"

Enter OpenAI API Key: ··········


In [ ]:
# cleaned_reviews.csv
# Load the dataset
DATASET_PATH = (
    DATA_DIR /
    "cleaned_reviews.csv"
)
reviews_df = pd.read_csv(
 DATASET_PATH
)
print(reviews_df.shape)
reviews_df.head()

(22600, 11)


,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [ ]:
# Inspect Dataset
print("=" * 60)
print("Dataset Summary")
print("=" * 60)

print(f"Rows      : {reviews_df.shape[0]:,}")
print(f"Columns   : {reviews_df.shape[1]}")

reviews_df.info()

Dataset Summary
Rows      : 22,600
Columns   : 11
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22600 entries, 0 to 22599
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Unnamed: 0               22600 non-null  int64 
 1   Clothing ID              22600 non-null  int64 
 2   Age                      22600 non-null  int64 
 3   Title                    19648 non-null  object
 4   Review Text              22600 non-null  object
 5   Rating                   22600 non-null  int64 
 6   Recommended IND          22600 non-null  int64 
 7   Positive Feedback Count  22600 non-null  int64 
 8   Division Name            22587 non-null  object
 9   Department Name          22587 non-null  object
 10  Class Name               22587 non-null  object
dtypes: int64(6), object(5)
memory usage: 1.9+ MB


In [ ]:
reviews_df.isnull().sum()

,0
Unnamed: 0,0
Clothing ID,0
Age,0
Title,2952
Review Text,0
Rating,0
Recommended IND,0
Positive Feedback Count,0
Division Name,13
Department Name,13


In [ ]:
reviews_df = reviews_df.drop_duplicates()

reviews_df = reviews_df.dropna(
    subset=["Review Text"]
)

reviews_df = reviews_df[
    reviews_df["Review Text"]
    .str.strip()
    .ne("")
]

reviews_df = reviews_df.reset_index(drop=True)

print(reviews_df.shape)

(22600, 11)


In [ ]:
# Save the clean dataset
CLEAN_DATASET =   (DATA_DIR /
    "cleaned_reviews.csv")

reviews_df.to_csv(
    CLEAN_DATASET,
    index=False
)

print("Saved!")

Saved!


In [ ]:
# Full dataset
# all_review_texts = reviews["Review Text"].tolist()

# Test subset
review_texts = reviews_df["Review Text"].head(100).tolist()

# print(f"Full dataset: {len(all_review_texts)} reviews")
print(f"Test dataset: {len(review_texts)} reviews")

Test dataset: 100 reviews


In [ ]:
# for continuation you can skip this am loading from google drive itself...
from pathlib import Path
# Root project folder
PROJECT_ROOT = Path("/content/drive/MyDrive/AIEngineering/WomenClothReview")

# Dataset folder
DATA_DIR = PROJECT_ROOT / "datasets"

DATASET_PATH = (
    DATA_DIR /
    "cleaned_reviews.csv"
)
reviews_df = pd.read_csv(
 DATASET_PATH
)
print(reviews_df.shape)
reviews_df.head()

(22600, 11)


,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [ ]:
import os
import time
import pickle
import logging

from pathlib import Path
from typing import List

from tqdm.auto import tqdm
from openai import OpenAI, RateLimitError

In [ ]:
# Configuration setting
from pathlib import Path

# Root project folder
PROJECT_ROOT = Path("/content/drive/MyDrive/AIEngineering/WomenClothReview")

# Dataset folder
DATA_DIR = PROJECT_ROOT / "datasets"

# Output folder
OUTPUT_DIR = PROJECT_ROOT / "outputs"

# Chroma database
VECTOR_DB_DIR = PROJECT_ROOT / "vectordb"

# Figures
FIGURES_DIR = OUTPUT_DIR / "figures"

EMBEDDING_MODEL = "text-embedding-3-small"

BATCH_SIZE = 10

MAX_RETRIES = 5

SLEEP_TIME = 3

CHECKPOINT_INTERVAL = 10

# OUTPUT_DIR should already exist from your setup
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_FILE = CHECKPOINT_DIR / "embedding_checkpoint.pkl"

EMBEDDING_FILE = OUTPUT_DIR / "review_embeddings.pkl"

In [ ]:
client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)
print(client)

In [ ]:
review_texts = (
    reviews_df["Review Text"]
    .head(20)
    .astype(str)
    .str.strip()
    .tolist()
)

print(len(review_texts))

20


In [ ]:
for review in review_texts[:5]:
    print(review)
    print("-" * 80)

Absolutely wonderful - silky and sexy and comfortable
--------------------------------------------------------------------------------
Love this dress!  it's sooo pretty.  i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite.  i bought a petite and am 5'8".  i love the length on me- hits just a little below the knee.  would definitely be a true midi on someone who is truly petite.
--------------------------------------------------------------------------------
I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly in

#Cost Estimation

In [ ]:
!pip install -q tiktoken

In [ ]:
import tiktoken

encoding = tiktoken.encoding_for_model(
    EMBEDDING_MODEL
)

total_tokens = sum(
    len(encoding.encode(text))
    for text in review_texts
)

print(total_tokens)

7622


In [ ]:
# Estimate Cost
PRICE_PER_MILLION = 0.02

estimated_cost = (
    total_tokens / 1_000_000
) * PRICE_PER_MILLION

print(
    f"Estimated Cost: ${estimated_cost:.6f}"
)

Estimated Cost: $0.000152


In [ ]:
# Save Token Report
# Root project folder
PROJECT_ROOT = Path("/content/drive/MyDrive/AIEngineering/WomenClothReview")

# Output folder
OUTPUT_DIR = PROJECT_ROOT / "outputs"
token_report = {
    "model": EMBEDDING_MODEL,
    "reviews": len(review_texts),
    "tokens": total_tokens,
    "estimated_cost": estimated_cost
}

with open(
    OUTPUT_DIR / "token_usage.json",
    "w"
) as f:
    json.dump(
        token_report,
        f,
        indent=4
    )

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

logger = logging.getLogger(__name__)

In [ ]:
# Create Embedding Function
def generate_embeddings(
    texts: List[str],
    model: str = EMBEDDING_MODEL,
) -> List[List[float]]:
    """
    Generate embeddings for a batch of texts.
    """

    response = client.embeddings.create(
        model=model,
        input=texts,
    )

    return [
        item.embedding
        for item in response.data
    ]

In [ ]:
def save_checkpoint(
    embeddings,
    processed_count,
):
    checkpoint = {
        "processed_count": processed_count,
        "embeddings": embeddings,
    }

    with open(CHECKPOINT_FILE, "wb") as f:
        pickle.dump(checkpoint, f)

    logger.info(
        "Checkpoint saved (%d embeddings).",
        processed_count,
    )


In [ ]:
def load_checkpoint():

    if not CHECKPOINT_FILE.exists():

        logger.info(
            "No checkpoint found. Starting from scratch."
        )

        return [], 0

    with open(CHECKPOINT_FILE, "rb") as f:

        checkpoint = pickle.load(f)

    logger.info(
        "Checkpoint loaded (%d embeddings).",
        checkpoint["processed_count"],
    )

    return (
        checkpoint["embeddings"],
        checkpoint["processed_count"],
    )

In [ ]:
# Unit test of the work
from unittest.mock import MagicMock

# Mock response objects
mock_embedding = MagicMock()
mock_embedding.embedding = [0.1, 0.2, 0.3]

mock_response = MagicMock()
mock_response.data = [mock_embedding]

# Mock the client
client.embeddings.create = MagicMock(return_value=mock_response)

# Call function
result = generate_embeddings(["Hello world"])

# Assertions
assert isinstance(result, list)
assert len(result) == 1
assert result == [[0.1, 0.2, 0.3]]

# Verify API call
client.embeddings.create.assert_called_once_with(
    model=EMBEDDING_MODEL,
    input=["Hello world"],
)

print("✅ All tests passed!")

✅ All tests passed!


In [ ]:
vectors = generate_embeddings(
    review_texts[:2]
)
len(vectors)
assert len(vectors)==1
print("✅ All tests passed!")

✅ All tests passed!


In [ ]:
!pip install -q tqdm

In [ ]:
# =====================================================
# Batch Embedding Pipeline
# =====================================================

def batch_embeddings(
    texts: List[str],
    batch_size: int = BATCH_SIZE,
) -> List[List[float]]:
    """
    Generate embeddings in batches.

    Supports:

    • Retry on rate limits
    • Resume from checkpoint
    • Progress bar
    • Logging

    Returns
    -------
    List of embedding vectors.
    """

    embeddings, processed_count = load_checkpoint()

    start_batch = processed_count // batch_size

    total_batches = (
        len(texts) + batch_size - 1
    ) // batch_size

    logger.info(
        "Starting embedding generation..."
    )

    progress_bar = tqdm(
        total=total_batches,
        initial=start_batch,
        desc="Embedding Reviews",
    )

    for batch_number in range(
        start_batch,
        total_batches,
    ):

        start = batch_number * batch_size

        batch = texts[
            start:
            start + batch_size
        ]

        retries = 0

        while True:

            try:

                batch_vectors = generate_embeddings(
                    batch
                )

                embeddings.extend(batch_vectors)

                processed_count += len(batch)

                logger.info(
                    "Batch %d/%d completed (%d/%d reviews).",
                    batch_number + 1,
                    total_batches,
                    processed_count,
                    len(texts),
                )

                if (
                    (batch_number + 1)
                    % CHECKPOINT_INTERVAL
                    == 0
                ):
                    save_checkpoint(
                        embeddings,
                        processed_count,
                    )

                progress_bar.update(1)

                time.sleep(SLEEP_TIME)

                break

            except RateLimitError:

                retries += 1

                if retries > MAX_RETRIES:

                    logger.error(
                        "Maximum retries exceeded."
                    )

                    raise

                wait = (
                    SLEEP_TIME
                    * (2 ** retries)
                )

                logger.warning(
                    "Rate limit reached. "
                    "Retry %d/%d in %d seconds.",
                    retries,
                    MAX_RETRIES,
                    wait,
                )

                time.sleep(wait)

            except Exception:

                logger.exception(
                    "Unexpected error."
                )

                save_checkpoint(
                    embeddings,
                    processed_count,
                )

                progress_bar.close()

                raise

    progress_bar.close()

    save_checkpoint(
        embeddings,
        processed_count,
    )

    with open(
        EMBEDDING_FILE,
        "wb",
    ) as f:

        pickle.dump(
            embeddings,
            f,
        )

    logger.info(
        "Embeddings saved to %s",
        EMBEDDING_FILE,
    )

    if CHECKPOINT_FILE.exists():

        CHECKPOINT_FILE.unlink()

        logger.info(
            "Checkpoint removed."
        )

    logger.info(
        "Embedding generation completed successfully."
    )

    return embeddings

In [ ]:
# =====================================================
# Run Pipeline
# =====================================================

embeddings = batch_embeddings(
    review_texts,
    batch_size=BATCH_SIZE,
)

logger.info(
    "Total Embeddings: %d",
    len(embeddings),
)

logger.info(
    "Embedding Dimension: %d",
    len(embeddings[0]),
)

Embedding Reviews:   0%|          | 0/2 [00:00<?, ?it/s]

#### Reload Embeddings

In [ ]:
with open(embedding_path, "rb") as f:
    embeddings = pickle.load(f)